In [1]:
# imports
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from transformers import pipeline
from datetime import datetime, timedelta
from tiingo import TiingoClient
from dotenv import load_dotenv
import os
from edgar import Company, set_identity
import regex as re
import torch
import time
import sys
sys.path.insert(0, '../../scripts')
import fnspid_news as fn


ModuleNotFoundError: No module named 'fnspid_news'

In [ ]:
import os
print(os.getcwd())

/Users/heiditam/Documents/summer_2026/factor_model/notebooks/cleaned


### Gathering Data

In [ ]:
headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
html = requests.get(url, headers=headers).text
table = pd.read_html(html)[0]
table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
table.head()

/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_29944/4004594965.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(html)[0]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


#### Tickers
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom), 
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalers**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [ ]:
# list of all the tickers with a dictionary that sorts them by type of stock
tickers = pd.read_csv('../../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
all_tickers = [t for group in ai_tickers.values() for t in group]

In [ ]:
# Data from 2018 - 2026 (pre-COVID to present)
prices = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start='2018-01-01',
    end='2026-01-01',
    auto_adjust=True,
    progress=False
)
prices.to_csv('../../data/prices_raw.csv')

In [ ]:
prices.head()

Price      Close                                                          \
Ticker        AI    AMD       AMZN       ANET ARM        ASML       AVGO   
Date                                                                       
2018-01-02   NaN  10.98  59.450500  14.439375 NaN  163.474472  21.015074   
2018-01-03   NaN  11.55  60.209999  14.725000 NaN  164.716141  21.244890   
2018-01-04   NaN  12.12  60.479500  14.543125 NaN  166.252213  21.251980   
2018-01-05   NaN  11.88  61.457001  14.798125 NaN  168.202148  21.377905   
2018-01-08   NaN  12.28  62.343498  15.691250 NaN  169.085175  21.429068   

Price                                        ...    Volume                     \
Ticker           DELL      GOOGL        HPE  ...      META     MRVL      MSFT   
Date                                         ...                                
2018-01-02  21.014696  53.154690  11.271499  ...  18151900  5619900  22483800   
2018-01-03  21.236492  54.061569  11.340933  ...  16886600  9014000  26061400   
2018-01-04  21.613802  54.271564  11.618670  ...  13880900  9221200  21912000   
2018-01-05  21.771864  54.991222  11.610958  ...  13574500  5855400  23407100   
2018-01-08  21.766764  55.185375  11.433510  ...  17994700  5529500  22113000   

Price                                                                 
Ticker           NVDA PLTR       QQQ     SMCI SOUN     SOXX      TSM  
Date                                                                  
2018-01-02  355616000  NaN  32573300  4344000  NaN  2860200  4984000  
2018-01-03  914704000  NaN  29383600  4094000  NaN  1326900  6963200  
2018-01-04  583268000  NaN  24776100  3631000  NaN  1335000  4876600  
2018-01-05  580124000  NaN  26992300  2178000  NaN  1170600  5330800  
2018-01-08  881216000  NaN  23159100  3484000  NaN  1622700  3538200  

[5 rows x 105 columns]

### Cleaning Data

In [ ]:
close  = prices['Close']
volume = prices['Volume']
soxx   = close['SOXX']
qqq    = close['QQQ']

In [ ]:
# Remove tickers with more than 5% NaN values
clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices.to_csv('../../data/clean_prices.csv')

In [ ]:
# Convert prices to log
log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()

### Feature Engineering

In [ ]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility: std of returns
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    # whether a stock has been moving up or down more forcefully over its recent trading days
    # > 70: overbought (a lot of upward activity)
    # < 30: oversold (a lot of downward activity)
    # 50: roughly equal up and down movement
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean() # average of up days
    loss  = (-delta.clip(upper=0)).rolling(14).mean() # average of down days
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [ ]:
all_feature_dfs = []
for ticker in all_tickers:
    if ticker not in close.columns:
        continue
    ticker_feats = make_features(
        close  = close[ticker],
        volume = volume[ticker],
        soxx   = soxx,
        qqq    = qqq
    )
    ticker_feats['Ticker'] = ticker      
    all_feature_dfs.append(ticker_feats)

In [ ]:
features_df = pd.concat(all_feature_dfs).reset_index()
features_df = features_df.set_index(['Date','Ticker'])

In [ ]:
# Forward 5-day cumulative return
features_df['fwd_5d'] = (
    features_df.groupby('Ticker')['ret_1d']
    .transform(lambda x: x.rolling(5).sum().shift(-5))
)

features_df = features_df.dropna(subset=['fwd_5d'])

# Cross-sectional: top half = 1, bottom half = 0 (guaranteed 50/50 per date)
# Which stocks will outperform their peers? 
features_df['Target'] = (
    features_df.groupby('Date')['fwd_5d']
    .transform(lambda x: (x > x.median()).astype(int))
)

In [ ]:
features_df.head()

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,fwd_5d,Target
Date,Ticker,,,,,,,,,,,,,,
2019-01-02,NVDA,0.020375,0.038500,0.071923,-0.051261,-0.166494,0.023554,0.034328,37.207850,-0.107088,-0.083558,0.785101,-0.528766,0.051363,1
2019-01-03,NVDA,-0.060417,-0.042350,-0.038392,-0.128964,-0.247295,0.033031,0.034169,30.211065,-0.110360,-0.119161,1.104590,-0.557237,0.130366,1
2019-01-04,NVDA,0.064068,0.020150,0.038271,-0.016749,-0.133155,0.045219,0.035031,40.655533,-0.080480,-0.078587,0.933377,-0.528870,0.091086,1
2019-01-07,NVDA,0.052941,0.052709,0.072952,0.061436,-0.094068,0.049557,0.037384,48.847620,-0.063841,-0.044049,1.128762,-0.503928,0.048963,1
2019-01-08,NVDA,-0.024895,0.092507,0.047416,0.079185,-0.052706,0.052560,0.034712,48.297140,-0.055655,-0.044008,1.240790,-0.516278,0.070070,1


In [ ]:
features_df.to_csv('../../data/features_df.csv')

### 500 Most Recent News Sentiment with FNSPID News (1999-2023)

In [ ]:
finbert = pipeline('sentiment-analysis', model='ProsusAI/finbert', truncation=True, max_length=512, device=-1)

Device set to use cpu


In [ ]:
def clean_filing_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)          # remove HTML tags
    text = re.sub(r'&[a-z]+;', ' ', text)          # remove HTML entities
    text = re.sub(r'\s+', ' ', text)               # normalize whitespace
    text = re.sub(r'\d{1,2}/\d{1,2}/\d{2,4}', '', text)  # remove dates
    text = re.sub(r'\$[\d,\.]+', '', text)         # remove dollar amounts
    text = re.sub(r"['']", '', text)               # drop apostrophes
    text = re.sub(r'[^\w\s\.]', ' ', text)         # remove special characters
    text = re.sub(r'\s{2,}', ' ', text)            # normalize whitespace again
    return text.strip()

In [ ]:
# imports
FNSPID_DIR = '../../data/fnspid'
RAW      = f'{FNSPID_DIR}/nasdaq_exteral_data.csv'
FILTERED = f'{FNSPID_DIR}/news_19.parquet'
DAILY    = f'{FNSPID_DIR}/news_daily_features.csv'

In [ ]:
# filter to 19 tickers
if not os.path.exists(FILTERED):
    news = fn.filter_news(raw=RAW, out=FILTERED)
else:
    news = pd.read_parquet(FILTERED)
    fn.coverage_report(news)

scanned 6,500,000 rows, kept 65,643
saved 63,632 articles -> ../../data/fnspid/news_19.parquet

Articles per ticker per year:
 published  2019  2020  2021  2022  2023  2024
Ticker                                       
NVDA        507   309   702  2332  5537     0
AMD         872  1054   805  1849  2127    70
INTC       2612  1537  1185  1379  2075     0
ASML        311   212   206   330   563     0
TSM         641   451   437   699   941     0
AVGO       1078   301     0     0     7     0
MRVL        503   259   180   242   418     0
ARM           0     0     0     0     0     0
SMCI          1    21    30   169   444     0
MSFT          0     0     0  2606  5910     0
GOOGL      2329  2374  1472  1876  2307     0
AMZN          0   277     0     0  4670     0
META          0   432     0     0     0     0
PLTR          0     0     0     0     0     0
AI          245   213   158   146   742     0
SOUN          0     0     0     0     0     0
DELL        113   161   178   272   413     0

In [ ]:
if not os.path.exists(DAILY):
    news_daily = fn.score_and_aggregate(filtered=FILTERED, out=DAILY)
else:
    news_daily = pd.read_csv(DAILY, parse_dates=['Date'])
news_daily.head()

Device set to use mps:0


scored 63,632/63,632
saved 13,167 ticker-days -> ../../data/fnspid/news_daily_features.csv


,Date,Ticker,news_sentiment,news_count,pct_positive
0,2018-12-31,AMD,0.000000,1,0.000000
1,2018-12-31,AVGO,0.000000,1,0.000000
2,2018-12-31,INTC,0.000000,1,0.000000
3,2019-01-01,AI,-0.882742,1,0.000000
4,2019-01-01,AMD,0.302095,3,0.333333


In [ ]:
# merge onto price features
features_df.index = features_df.index.set_levels(
    pd.to_datetime(features_df.index.levels[0]), level='Date')   # make sure Date is datetime
df = fn.merge_news(features_df, daily_path=DAILY, news_end='2023-12-31')

news_cols = ['news_sentiment', 'news_count', 'pct_positive']
print(df[news_cols].notna().groupby(df.index.get_level_values('Date').year).mean().round(2))

      news_sentiment  news_count  pct_positive
Date                                          
2019             1.0         1.0           1.0
2020             1.0         1.0           1.0
2021             1.0         1.0           1.0
2022             1.0         1.0           1.0
2023             1.0         1.0           1.0
2024             0.0         0.0           0.0
2025             0.0         0.0           0.0


### SEC 8K Filings (Earnings Releases)

In [ ]:
# Domain keyword dictionaries
NVIDIA_KEYWORDS = [
    'h100', 'h200', 'h20',
    'blackwell', 'b100', 'b200', 'gb200', 'gb300',
    'rubin', 'vera rubin', 'nvl72',
    'dgx', 'hgx',
    'cuda', 'tensorrt', 'triton', 'nim microservices',
    'hopper architecture', 'blackwell architecture',
    
]

AMD_KEYWORDS = [
    'instinct mi300', 'instinct mi300x', 'instinct mi325',
    'instinct mi350', 'instinct mi400',
    'instinct mi250', 'instinct mi200',
    'radeon pro', 'radeon instinct',
    'rocm', 'hip framework', 'amd cdna',
    'amd rdna', 'amd infinity fabric',
    'amd gpu demand', 'amd accelerator',
    'amd data center gpu', 'amd ai accelerator',
    'amd compute demand', 'amd inference',
    'amd training workload', 'amd cluster',
]

INTEL_KEYWORDS = [
    'gaudi2', 'gaudi 2', 'gaudi3', 'gaudi 3',
    'habana labs', 'habana gaudi',
    'ponte vecchio', 'max series gpu',
    'intel gpu max', 'flex series gpu',
    'openvino', 'intel oneapi', 'intel neural compressor',
    'intel xeon ai', 'intel core ultra ai',
    'meteor lake ai', 'lunar lake ai', 'arrow lake ai', 'intel npu',
    'intel data center gpu', 'intel ai accelerator',
    'intel compute demand', 'intel ai inference',
    'intel training workload', 'gaudi deployment', 'gaudi adoption', 'gaudi demand',
]

GENERAL_GPU_KEYWORDS = [
    'gpu demand', 'gpu supply', 'gpu allocation',
    'data center revenue', 'data center demand',
    'inference demand', 'inference workload',
    'training compute', 'compute demand',
    'cloud ai', 'ai infrastructure', 'ai accelerator',
    'tensor processing', 'custom ai chip',
]


CAPEX_UP_KEYWORDS = [
    'increasing capex', 'raised capex', 'accelerating capex',
    'accelerating investment', 'capital expenditure increase',
    'expanding capacity', 'record investment',
    'investing heavily', 'multiyear investment',
    'committed to investing', 'significant investment in ai',
    'capital investment', 'infrastructure investment', 'ai investment'
]

CAPEX_DOWN_KEYWORDS = [
    'reducing capex', 'cutting investment', 'slowing spend',
    'deferring investment', 'pausing investment',
    'capital efficiency', 'disciplined spending',
    'revisiting capex', 'moderating investment'
]

COMPETITIVE_RISK_KEYWORDS = [
    'amd instinct', 'gaudi', 'trainium', 'inferentia', 'maia',
    'google tpu', 'axion',
    'custom silicon', 'in-house chip', 'in-house ai chip',
    'merchant silicon', 'proprietary chip',
    'competitive alternative', 'alternative to nvidia',
    'amd alternative', 'nvidia alternative',
    'vendor diversification', 'multi vendor gpu',
]

In [ ]:
def score_sentences_finbert(sentences, max_sentences=20):
    """Score a list of sentences with FinBERT, return mean sentiment"""
    if not sentences:
        return 0.0, 0.0, 0
    
    scored = []
    for sent in sentences[:max_sentences]:
        try:
            # Truncate long sentences to 512 chars
            result = finbert(sent[:512])[0]
            score = result['score'] if result['label'] == 'positive' \
                   else -result['score'] if result['label'] == 'negative' \
                   else 0.0
            scored.append(score)
        except Exception:
            continue
    
    if not scored:
        return 0.0, 0.0, 0
    
    mean_sentiment = np.mean(scored)
    positive_ratio = sum(1 for s in scored if s > 0) / len(scored)
    return mean_sentiment, positive_ratio, len(scored)

In [ ]:
def extract_domain_signals(transcript_text):
    text_lower = transcript_text.lower()
    signals = {}

    # Raw mention counts (normalize by transcript length)
    word_count = len(text_lower.split())
    signals['nvidia_mentions'] = sum(text_lower.count(k) for k in NVIDIA_KEYWORDS) / word_count * 1000
    signals['amd_mentions'] = sum(text_lower.count(k) for k in AMD_KEYWORDS) / word_count * 1000
    signals['intel_mentions'] = sum(text_lower.count(k) for k in INTEL_KEYWORDS) / word_count * 1000
    signals['gpu_mentions'] = sum(text_lower.count(k) for k in GENERAL_GPU_KEYWORDS) / word_count * 1000
    signals['capex_up_score']  = sum(text_lower.count(k) for k in CAPEX_UP_KEYWORDS) / word_count * 1000
    signals['capex_down_score']= sum(text_lower.count(k) for k in CAPEX_DOWN_KEYWORDS) / word_count * 1000
    competitor_keys = [k for k in COMPETITIVE_RISK_KEYWORDS if k != ticker.lower()]
    signals['competitor_mentions'] = sum(text_lower.count(k) for k in competitor_keys) / word_count * 1000
    signals['capex_net']       = signals['capex_up_score'] - signals['capex_down_score'] 

    # Paragraph-level: find sentences mentioning AI demand
    ai_sentences = [s for s in transcript_text.split('.')
                    if any(k in s.lower() for k in ['artificial intelligence','ai demand','ai revenue', 'ai growth', 'ai contribution'])]
    signals['ai_sentence_ratio'] = len(ai_sentences) / max(len(transcript_text.split('.')),1)

    return signals

In [ ]:
load_dotenv()

True

In [ ]:
set_identity(os.getenv('EMAIL'))

In [ ]:
# get earnings transcript 
def get_earnings_transcripts(ticker, start_date='2018-01-01', end_date=None):
    if end_date is None:
        end_date = datetime.today().strftime('%Y-%m-%d')
    company = Company(ticker)
    filings = company.get_filings(form="8-K", filing_date=f'{start_date}:{end_date}')
    
    results = []
    for filing in filings:
        try:
            # only earnings releases
            if '2.02' not in str(filing.items):
                continue
            
            for attachment in filing.attachments:
                if 'EX-99' not in attachment.document_type:
                    continue
                
                text = attachment.text()

                if not text or len(text) < 200:
                    continue
                
                text_after = clean_filing_text(text)
                
                signals = extract_domain_signals(text_after)
                
                results.append({
                    'Ticker': ticker,
                    'Date': pd.to_datetime(filing.filing_date),
                    'Text': text_after,
                    **signals
                })
                break  # one exhibit per filing
                
        except Exception as e:
            print(f"Skipped {ticker} {filing.filing_date}: {e}")
            continue
    
    print(f"{ticker}: {len(results)} qualifying filings")
    return results

In [ ]:
all_transcript_signals = []

for ticker in all_tickers:
    transcripts = get_earnings_transcripts(ticker)
    for t in transcripts:
        signals = extract_domain_signals(t['Text'])
        signals['Ticker'] = t['Ticker']
        signals['Date'] = t['Date']
        all_transcript_signals.append(signals)

transcript_signals_df = (pd.DataFrame(all_transcript_signals).set_index(['Date', 'Ticker']).sort_index())

transcript_signals_df.to_csv('../../data/transcript_signals_df.csv')
transcript_signals_df.head()

NVDA: 37 qualifying filings


AMD: 37 qualifying filings


INTC: 36 qualifying filings
ASML: 0 qualifying filings
TSM: 0 qualifying filings


AVGO: 34 qualifying filings


MRVL: 24 qualifying filings
ARM: 0 qualifying filings


...

SOUN: 19 qualifying filings


DELL: 35 qualifying filings


ANET: 35 qualifying filings


HPE: 44 qualifying filings


nvidia_mentions  amd_mentions  intel_mentions  gpu_mentions  capex_up_score  capex_down_score  competitor_mentions  capex_net  ai_sentence_ratio
Date       Ticker                                                                                                                                                  
2018-01-25 INTC                0.0      0.000000             0.0           0.0        0.000000               0.0                  0.0   0.000000           0.003333
2018-01-30 AMD                 0.0      0.366233             0.0           0.0        0.000000               0.0                  0.0   0.000000           0.000000
           SMCI                0.0      0.000000             0.0           0.0        0.809061               0.0                  0.0   0.809061           0.000000
2018-01-31 META                0.0      0.000000             0.0           0.0        0.000000               0.0                  0.0   0.000000           0.000000
           MSFT                0.0      0.000000             0.0           0.0        0.000000               0.0                  0.0   0.000000           0.000000

In [ ]:
nvda_tickers = get_earnings_transcripts('NVDA')
nvda_tickers[0]

NVDA: 37 qualifying filings


{'Ticker': 'NVDA',
 'Date': Timestamp('2026-08-26 00:00:00'),
 'Text': 'NVIDIA Announces Financial Results for Second Quarter Fiscal 2027 Revenue of billion up 106 from a year ago Data Center revenue of billion up 117 from a year ago SANTA CLARA Calif. Aug. 26 2026 NVIDIA NASDAQ NVDA today reported revenue for the second quarter ended July 26 2026 of billion up 18 from the previous quarter and up 106 from a year ago. For the quarter GAAP and non GAAP gross margins were both 75.0 . GAAP and non GAAP earnings per diluted share were and respectively. AI has reached its inflection point. It s doing useful work. Its tokens are productive and profitable. Now compute is revenue said Jensen Huang founder and CEO of NVIDIA. And demand is accelerating. This time last year one lab alone was driving the buildout today we have a golden age of new AI labs and startups multiple frontier labs scaling in parallel a thriving open model ecosystem and physical AI coming online with strong momentum across 

In [ ]:
import textwrap
print(textwrap.fill(nvda_tickers[0]['Text'][:2000], width=100))

NVIDIA Announces Financial Results for Second Quarter Fiscal 2027 Revenue of billion up 106 from a
year ago Data Center revenue of billion up 117 from a year ago SANTA CLARA Calif. Aug. 26 2026
NVIDIA NASDAQ NVDA today reported revenue for the second quarter ended July 26 2026 of billion up 18
from the previous quarter and up 106 from a year ago. For the quarter GAAP and non GAAP gross
margins were both 75.0 . GAAP and non GAAP earnings per diluted share were and respectively. AI has
reached its inflection point. It s doing useful work. Its tokens are productive and profitable. Now
compute is revenue said Jensen Huang founder and CEO of NVIDIA. And demand is accelerating. This
time last year one lab alone was driving the buildout today we have a golden age of new AI labs and
startups multiple frontier labs scaling in parallel a thriving open model ecosystem and physical AI
coming online with strong momentum across the U.S. and around the world. The AI infrastructure
buildout is at full

In [ ]:
# merge back to original dataframe
df = df.reset_index()

df = pd.merge_asof(
    df.sort_values('Date'),
    transcript_signals_df.reset_index().rename(columns={'date': 'Date'}).sort_values('Date'),
    on='Date',
    by='Ticker',
    direction='backward'
)

with_domain_keywords = df.set_index(['Date', 'Ticker'])
with_domain_keywords[['gpu_mentions','capex_up_score','capex_down_score', 'capex_net','competitor_mentions','ai_sentence_ratio']] = \
    with_domain_keywords[['gpu_mentions','capex_up_score','capex_down_score', 'capex_net','competitor_mentions','ai_sentence_ratio']].fillna(0)

### Final Cleaning

In [ ]:
edgar_cols = ['nvidia_mentions', 'amd_mentions', 'intel_mentions',
            'gpu_mentions', 'capex_up_score', 'capex_down_score', 
              'competitor_mentions', 'capex_net', 'ai_sentence_ratio']

In [ ]:
with_domain_keywords[edgar_cols] = with_domain_keywords[edgar_cols].fillna(0)

In [ ]:
# Lag one trading day: earnings 8-Ks are often filed after the close
with_domain_keywords = with_domain_keywords.sort_index(level=['Ticker', 'Date'])
with_domain_keywords[edgar_cols] = (
    with_domain_keywords.groupby(level='Ticker')[edgar_cols].shift(1).fillna(0)
)

In [ ]:
with_domain_keywords.to_csv('../../data/ai_sector_features_news_v2.csv')

In [ ]:
with_domain_keywords.head()

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,...,pct_positive,nvidia_mentions,amd_mentions,intel_mentions,gpu_mentions,capex_up_score,capex_down_score,competitor_mentions,capex_net,ai_sentence_ratio
Date,Ticker,,,,,,,,,,,,,,,,,,,,,
2021-12-08,AI,0.061193,0.142521,0.004730,-0.088251,-0.331432,0.070865,0.051896,27.025095,-0.376055,-0.342284,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
2021-12-09,AI,-0.055899,0.060826,0.068242,-0.156192,-0.303300,0.049221,0.049631,29.816945,-0.354911,-0.314134,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006369
2021-12-10,AI,0.048302,0.050265,0.130756,-0.123502,-0.265823,0.049006,0.051682,36.949875,-0.305373,-0.284820,...,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006369
2021-12-13,AI,-0.038347,-0.048250,0.069421,-0.145310,-0.315634,0.057091,0.050789,38.270846,-0.317696,-0.309451,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006369
2021-12-14,AI,-0.036167,-0.028358,-0.026538,-0.156385,-0.354184,0.054528,0.050096,36.438244,-0.352169,-0.337940,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.006369


In [ ]:
missing = set(all_tickers) - set(transcript_signals_df.index.
    get_level_values('Ticker').unique())
print(missing)
# {'ARM', 'TSM', 'ASML'}

{'TSM', 'ARM', 'ASML'}
